# LeetCode 1738 — Find Kth Largest XOR Coordinate Value bằng Python

Link đề: https://leetcode.com/problems/find-kth-largest-xor-coordinate-value/ · Độ khó: **Medium** · Chủ đề: Array, Divide and Conquer, Bit Manipulation, Sorting, Heap (Priority Queue), Matrix, Prefix Sum, Quickselect

Skeleton LeetCode:

```python
class Solution:
    def kthLargestValue(self, matrix: List[List[int]], k: int) -> int:
        ...
```

Lộ trình: **tính trực tiếp từng tọa độ** bằng cách XOR cả hình chữ nhật (`O(m²n²)`) → **XOR tiền tố 2 chiều** (`O(mn)`) rồi **sắp xếp** → **min-heap kích thước `k`** (`O(mn log k)`) → **quickselect** (`O(mn)` trung bình).

## 1. Mục tiêu học

1. Hiểu "giá trị tọa độ `(a, b)`" = XOR của hình chữ nhật từ `(0, 0)` tới `(a, b)` — một **tiền tố 2 chiều**.
2. Chuyển công thức **tổng tiền tố 2D** (bao hàm – loại trừ) sang **XOR**: `P[i][j] = P[i−1][j] ^ P[i][j−1] ^ P[i−1][j−1] ^ M[i][j]` — "trừ" chính là XOR.
3. Giải bài toán **phần tử lớn thứ `k`** bằng sắp xếp, heap kích thước `k`, và quickselect.
4. Hiểu đánh đổi giữa `O(N log N)`, `O(N log k)` và `O(N)` trung bình với `N = m·n ≤ 10⁶`.
5. Kiểm chứng bằng tính trực tiếp trên ma trận nhỏ.

## 2. Đề bài & ví dụ

Cho ma trận `matrix` kích thước `m × n` gồm số nguyên không âm, và số nguyên `k`.

**Giá trị** của tọa độ `(a, b)` là XOR của mọi `matrix[i][j]` với `0 ≤ i ≤ a < m` và `0 ≤ j ≤ b < n` (chỉ số từ 0).

Tìm giá trị **lớn thứ `k`** (đánh số từ 1) trong số giá trị của **mọi** tọa độ.

**Ví dụ 1:** `matrix = [[5,2],[1,6]], k = 1` → `7` (tọa độ `(0,1)`: `5 ^ 2 = 7`).

**Ví dụ 2:** cùng ma trận, `k = 2` → `5` (tọa độ `(0,0)`).

**Ví dụ 3:** cùng ma trận, `k = 3` → `4` (tọa độ `(1,0)`: `5 ^ 1 = 4`).

**Ràng buộc:**
- `1 ≤ m, n ≤ 1000`
- `0 ≤ matrix[i][j] ≤ 10⁶`
- `1 ≤ k ≤ m · n`

In [1]:
import random
import time
import heapq
from typing import List

M = [[5, 2], [1, 6]]
examples = [((M, 1), 7), ((M, 2), 5), ((M, 3), 4)]
for (mat, k), want in examples:
    print(f"matrix={mat}, k={k} -> kỳ vọng {want}")

matrix=[[5, 2], [1, 6]], k=1 -> kỳ vọng 7
matrix=[[5, 2], [1, 6]], k=2 -> kỳ vọng 5
matrix=[[5, 2], [1, 6]], k=3 -> kỳ vọng 4


## 3. Kiến thức nền

### 3.1 XOR: "cộng" và "trừ" là một

`x ^ x = 0` ⇒ XOR là phép **tự nghịch đảo**: thêm một phần tử rồi thêm lại lần nữa thì nó biến mất. Vì vậy công thức bao hàm – loại trừ của tổng tiền tố chuyển sang XOR bằng cách thay cả `+` lẫn `−` bằng `^`.

In [2]:
print(5 ^ 2, 5 ^ 2 ^ 2, (5 ^ 2) ^ (2 ^ 6) ^ 2)

7 5 1


### 3.2 Tổng tiền tố 2D (nhắc lại)

`S[i][j] = S[i−1][j] + S[i][j−1] − S[i−1][j−1] + M[i][j]`: hình chữ nhật trên và hình chữ nhật trái chồng nhau ở góc trên-trái, bị cộng hai lần nên trừ đi một lần.

In [3]:
def prefix_sum_2d(mat):
    m, n = len(mat), len(mat[0])
    S = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m):
        for j in range(n):
            S[i + 1][j + 1] = S[i][j + 1] + S[i + 1][j] - S[i][j] + mat[i][j]
    return S

print(prefix_sum_2d([[1, 2], [3, 4]]))

[[0, 0, 0], [0, 1, 3], [0, 4, 10]]


### 3.3 XOR tiền tố 2D

Với XOR: phần chồng góc trên-trái bị XOR **hai lần** (triệt tiêu) ⇒ cần XOR thêm **một lần** để khôi phục:

`P[i][j] = P[i−1][j] ^ P[i][j−1] ^ P[i−1][j−1] ^ M[i][j]`.

In [4]:
def prefix_xor_2d(mat):
    m, n = len(mat), len(mat[0])
    P = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m):
        for j in range(n):
            P[i + 1][j + 1] = P[i][j + 1] ^ P[i + 1][j] ^ P[i][j] ^ mat[i][j]
    return P

P = prefix_xor_2d(M)
print("P (bỏ hàng/cột 0 đệm):", [row[1:] for row in P[1:]])

P (bỏ hàng/cột 0 đệm): [[5, 7], [4, 0]]


### 3.4 Phần tử lớn thứ `k`

- `sorted(vals, reverse=True)[k − 1]` — đơn giản, `O(N log N)`.
- `heapq.nlargest(k, vals)[-1]` hoặc tự giữ **min-heap kích thước `k`**: đỉnh heap là phần tử lớn thứ `k` trong những gì đã thấy — `O(N log k)`.
- **Quickselect**: phân hoạch quanh pivot ngẫu nhiên, chỉ đệ quy một phía — `O(N)` trung bình.

In [5]:
vals = [7, 1, 5, 3, 9, 4]
k = 2
print(sorted(vals, reverse=True)[k - 1], heapq.nlargest(k, vals)[-1])

7 7


## 4. Ý tưởng nền tảng / trực giác

Ví dụ `[[5,2],[1,6]]`:

| tọa độ | các phần tử XOR | giá trị |
|---|---|---|
| (0,0) | 5 | 5 |
| (0,1) | 5, 2 | 7 |
| (1,0) | 5, 1 | 4 |
| (1,1) | 5, 2, 1, 6 | 0 |

Sắp giảm dần: `7, 5, 4, 0` ⇒ `k = 1 → 7`, `k = 2 → 5`, `k = 3 → 4`, `k = 4 → 0`. ✓

Kiểm tra công thức XOR tiền tố cho `(1,1)`: `P(0,1) ^ P(1,0) ^ P(0,0) ^ M[1][1] = 7 ^ 4 ^ 5 ^ 6 = 0`. ✓

In [6]:
print(7 ^ 4 ^ 5 ^ 6)

0


## 5. Cách 1 — Brute Force / baseline: tính trực tiếp từng tọa độ

**Ý tưởng:** với mỗi `(a, b)`, XOR mọi phần tử trong hình chữ nhật `[0..a] × [0..b]`; thu thập mọi giá trị, sắp xếp, lấy phần tử thứ `k`.

In [7]:
def kth_brute(matrix, k):
    m, n = len(matrix), len(matrix[0])
    vals = []
    for a in range(m):
        for b in range(n):
            x = 0
            for i in range(a + 1):
                for j in range(b + 1):
                    x ^= matrix[i][j]
            vals.append(x)
    vals.sort(reverse=True)
    return vals[k - 1]

for (mat, k), want in examples:
    print(kth_brute(mat, k), "kỳ vọng", want)

7 kỳ vọng 7
5 kỳ vọng 5
4 kỳ vọng 4


**Độ phức tạp:** Time `Σ_{a,b} (a+1)(b+1) = O(m² n²)` — với `m = n = 1000` là `2.5·10¹¹`. Space `O(mn)`.

**Nút thắt:** các hình chữ nhật lồng nhau — tọa độ `(a, b)` có thể suy ra từ các tọa độ lân cận đã tính (tiền tố).

In [8]:
random.seed(0)
for s in [10, 20, 30]:
    mat = [[random.randint(0, 10**6) for _ in range(s)] for _ in range(s)]
    t0 = time.perf_counter(); kth_brute(mat, 1)
    print(f"{s}x{s}: {time.perf_counter() - t0:.3f}s")

10x10: 0.000s
20x20: 0.001s
30x30: 0.006s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (tiền tố 2D):** `P[a][b] = P[a−1][b] ^ P[a][b−1] ^ P[a−1][b−1] ^ M[a][b]`. Mỗi giá trị `O(1)` ⇒ mọi tọa độ `O(mn)`.

**Insight 2 (chọn thứ `k`):** không cần sắp toàn bộ `N = mn` giá trị:
- giữ **min-heap kích thước `k`**: mỗi giá trị mới lớn hơn đỉnh thì thay đỉnh ⇒ cuối cùng đỉnh là lớn thứ `k`;
- hoặc **quickselect** `O(N)` trung bình.

**Mẹo bộ nhớ:** có thể tính `P` theo từng hàng (chỉ giữ hàng trước) và XOR tiền tố của hàng hiện tại: `P[a][b] = P[a−1][b] ^ rowxor(a, 0..b)`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — XOR tiền tố 2D + sắp xếp

In [9]:
def kth_prefix_sort(matrix, k):
    m, n = len(matrix), len(matrix[0])
    P = [[0] * (n + 1) for _ in range(m + 1)]
    vals = []
    for i in range(m):
        row, up, cur = matrix[i], P[i], P[i + 1]
        for j in range(n):
            v = up[j + 1] ^ cur[j] ^ up[j] ^ row[j]
            cur[j + 1] = v
            vals.append(v)
    vals.sort(reverse=True)
    return vals[k - 1]

for (mat, k), want in examples:
    print(kth_prefix_sort(mat, k), "kỳ vọng", want)

7 kỳ vọng 7
5 kỳ vọng 5
4 kỳ vọng 4


**Độ phức tạp:** Time `O(mn + mn log(mn))`, Space `O(mn)`.

### 7.2 Cách 3 — Tiền tố theo hàng + min-heap kích thước `k`

Chỉ giữ một hàng `prev` (giá trị `P` của hàng trên). Với hàng `i`, giữ `run` = XOR của `matrix[i][0..j]`; khi đó `P[i][j] = prev[j] ^ run`.

In [10]:
def kth_heap(matrix, k):
    n = len(matrix[0])
    prev = [0] * n
    h = []
    for row in matrix:
        run = 0
        for j in range(n):
            run ^= row[j]
            v = prev[j] ^ run
            prev[j] = v
            if len(h) < k:
                heapq.heappush(h, v)
            elif v > h[0]:
                heapq.heapreplace(h, v)
    return h[0]

for (mat, k), want in examples:
    print(kth_heap(mat, k), "kỳ vọng", want)

7 kỳ vọng 7
5 kỳ vọng 5
4 kỳ vọng 4


**Độ phức tạp:** Time `O(mn log k)`, Space `O(n + k)`. Rất tốt khi `k` nhỏ; khi `k ≈ mn` thì giống sắp xếp.

### 7.3 Cách 4 — Quickselect (chọn thứ `k`) trên mảng giá trị

Phân hoạch ba nhánh quanh pivot ngẫu nhiên (`> pivot`, `= pivot`, `< pivot`), chỉ đi tiếp vào nhánh chứa phần tử thứ `k`. Viết **lặp** để tránh đệ quy sâu.

In [11]:
def quickselect_kth_largest(vals, k, rng=random.Random(1738)):
    arr = vals
    while True:
        pivot = arr[rng.randrange(len(arr))]
        big = [x for x in arr if x > pivot]
        if k <= len(big):
            arr = big
            continue
        eq = sum(1 for x in arr if x == pivot)
        if k <= len(big) + eq:
            return pivot
        k -= len(big) + eq
        arr = [x for x in arr if x < pivot]

def kth_quickselect(matrix, k):
    n = len(matrix[0])
    prev = [0] * n
    vals = []
    for row in matrix:
        run = 0
        for j in range(n):
            run ^= row[j]
            prev[j] ^= run
            vals.append(prev[j])
    return quickselect_kth_largest(vals, k)

for (mat, k), want in examples:
    print(kth_quickselect(mat, k), "kỳ vọng", want)

7 kỳ vọng 7
5 kỳ vọng 5
4 kỳ vọng 4


Dòng `prev[j] ^= run` tương đương `prev[j] = prev[j] ^ run` — cập nhật hàng `P` tại chỗ.

**Độ phức tạp:** Time `O(mn)` trung bình (mỗi vòng giảm kích thước kỳ vọng theo hằng số), `O((mn)²)` worst rất hiếm với pivot ngẫu nhiên. Space `O(mn)`.

## 8. TRACE từng bước

Trace Cách 3 trên `[[5,2],[1,6]]`, `k = 2`:

| hàng i | j | run (XOR hàng 0..j) | prev[j] cũ | P = prev ^ run | heap (k=2) |
|---|---|---|---|---|---|
| 0 | 0 | 5 | 0 | 5 | [5] |
| 0 | 1 | 7 | 0 | 7 | [5, 7] |
| 1 | 0 | 1 | 5 | 4 | 4 < 5 ⇒ [5, 7] |
| 1 | 1 | 7 | 7 | 0 | 0 < 5 ⇒ [5, 7] |

Đỉnh heap `5` ⇒ lớn thứ 2.

In [12]:
def trace_heap(matrix, k):
    n = len(matrix[0]); prev = [0] * n; h = []
    for i, row in enumerate(matrix):
        run = 0
        for j in range(n):
            run ^= row[j]; old = prev[j]; v = old ^ run; prev[j] = v
            action = ""
            if len(h) < k:
                heapq.heappush(h, v); action = "đẩy vào"
            elif v > h[0]:
                out = heapq.heapreplace(h, v); action = f"thay {out}"
            else:
                action = "bỏ qua (≤ đỉnh)"
            print(f"({i},{j}): run={run}, prev cũ={old}, P={v} -> {action:16s} heap={sorted(h)}")
    print("lớn thứ", k, "=", h[0])
    return h[0]

trace_heap([[5, 2], [1, 6]], 2)

(0,0): run=5, prev cũ=0, P=5 -> đẩy vào          heap=[5]
(0,1): run=7, prev cũ=0, P=7 -> đẩy vào          heap=[5, 7]
(1,0): run=1, prev cũ=5, P=4 -> bỏ qua (≤ đỉnh)  heap=[5, 7]
(1,1): run=7, prev cũ=7, P=0 -> bỏ qua (≤ đỉnh)  heap=[5, 7]
lớn thứ 2 = 5


5

In [13]:
mat = [[1, 3, 5], [2, 4, 6], [7, 8, 9]]
P = prefix_xor_2d(mat)
print("giá trị mọi tọa độ:")
for row in P[1:]:
    print(row[1:])
vals = sorted((v for row in P[1:] for v in row[1:]), reverse=True)
print("sắp giảm dần:", vals)
for k in [1, 3, 9]:
    print(f"k={k}: brute={kth_brute(mat, k)}, heap={kth_heap(mat, k)}, quickselect={kth_quickselect(mat, k)}")

giá trị mọi tọa độ:
[1, 2, 7]
[3, 4, 7]
[4, 11, 1]
sắp giảm dần: [11, 7, 7, 4, 4, 3, 2, 1, 1]
k=1: brute=11, heap=11, quickselect=11
k=3: brute=7, heap=7, quickselect=7
k=9: brute=1, heap=1, quickselect=1


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def kthLargestValue(self, matrix: List[List[int]], k: int) -> int:
        n = len(matrix[0])
        prev = [0] * n
        vals = []
        for row in matrix:
            run = 0
            for j in range(n):
                run ^= row[j]
                prev[j] ^= run
            vals.extend(prev)
        vals.sort(reverse=True)
        return vals[k - 1]

sol = Solution()
for (mat, k), want in examples:
    got = sol.kthLargestValue(mat, k)
    print(got, "OK" if got == want else "SAI")

7 OK
5 OK
4 OK


Bản nộp dùng sắp xếp (Timsort bằng C) vì trong Python nó thường **nhanh hơn** heap/quickselect viết bằng Python dù độ phức tạp lý thuyết lớn hơn. `vals.extend(prev)` sao chép cả hàng một lần — nhanh hơn `append` từng phần tử.

## 10. Vì sao thuật toán đúng

**XOR tiền tố 2D.** Gọi `R(a, b)` là tập ô `[0..a] × [0..b]`. Ta có `R(a, b) = R(a−1, b) Δ R(a, b−1) Δ R(a−1, b−1) Δ {(a, b)}` theo nghĩa **hiệu đối xứng** (mỗi ô thuộc một số lẻ các tập bên phải ⇔ thuộc `R(a, b)`): ô trong `R(a−1, b−1)` thuộc ba tập đầu (lẻ), ô ở hàng `a` (trừ góc) chỉ thuộc `R(a, b−1)`, ô ở cột `b` chỉ thuộc `R(a−1, b)`, ô `(a, b)` thuộc tập cuối. Vì XOR trên hiệu đối xứng chính là XOR các XOR thành phần, công thức đúng.

**Dạng theo hàng.** `P[a][b] = P[a−1][b] ^ (M[a][0] ^ … ^ M[a][b])` — hình chữ nhật `R(a, b)` = `R(a−1, b)` ∪ phần hàng `a` từ `0..b` (rời nhau).

**Heap kích thước `k`.** Bất biến: heap chứa `k` giá trị lớn nhất trong số đã xét (với lặp). Thêm `v`: nếu `v` lớn hơn đỉnh (nhỏ nhất trong `k` lớn nhất), `v` thay nó; ngược lại `v` không thuộc top-`k`. Cuối cùng đỉnh là lớn thứ `k`.

**Quickselect.** Sau khi phân hoạch, phần tử lớn thứ `k` nằm trong `big` nếu `k ≤ |big|`, bằng `pivot` nếu `k ≤ |big| + |eq|`, ngược lại là lớn thứ `k − |big| − |eq|` trong `small`.

## 11. Phân tích Time / Space

Gọi `N = m · n ≤ 10⁶`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Trực tiếp | `O(m² n²)` | `O(N)` | ma trận ≤ 30×30 |
| 2. Tiền tố 2D + sort | `O(N log N)` | `O(N)` | |
| 3. Tiền tố theo hàng + heap | `O(N log k)` | `O(n + k)` | tốt khi `k` nhỏ |
| 4. Quickselect | `O(N)` TB | `O(N)` | lý thuyết tốt nhất |
| Nộp | `O(N log N)` | `O(N)` | sort bằng C, nhanh thực tế |

In [15]:
random.seed(1)
m = n = 500
mat = [[random.randint(0, 10**6) for _ in range(n)] for _ in range(m)]
for k in [1, 1000, m * n // 2]:
    line = f"k={k:6d}:"
    res = []
    for name, fn in [("sort", kth_prefix_sort), ("heap", kth_heap), ("qsel", kth_quickselect), ("submit", Solution().kthLargestValue)]:
        t0 = time.perf_counter(); r = fn(mat, k); res.append(r)
        line += f" {name}={time.perf_counter() - t0:.3f}s"
    assert len(set(res)) == 1
    print(line, "| giá trị =", res[0])

k=     1: sort=0.069s heap=0.019s qsel=0.030s submit=0.057s | giá trị = 1048571
k=  1000: sort=0.065s heap=0.022s qsel=0.031s submit=0.056s | giá trị = 1044210


k=125000: sort=0.066s heap=0.089s qsel=0.066s submit=0.056s | giá trị = 522679


## 12. Bộ test

In [16]:
fns = [kth_prefix_sort, kth_heap, kth_quickselect, lambda a, k: Solution().kthLargestValue(a, k)]

for (mat, k), want in examples:
    assert kth_brute(mat, k) == want
    for fn in fns:
        assert fn(mat, k) == want
assert Solution().kthLargestValue(M, 4) == 0

edge = [(([[0]], 1), 0), (([[10**6]], 1), 10**6), (([[1, 1, 1]], 2), 1), (([[1], [1], [1]], 3), 0),
        (([[0, 0], [0, 0]], 4), 0)]
for (mat, k), want in edge:
    assert kth_brute(mat, k) == want, (mat, k)
    for fn in fns:
        assert fn(mat, k) == want, (fn, mat, k)

random.seed(2024)
for _ in range(500):
    m, n = random.randint(1, 6), random.randint(1, 6)
    hi = random.choice([1, 7, 10**6])
    mat = [[random.randint(0, hi) for _ in range(n)] for _ in range(m)]
    k = random.randint(1, m * n)
    want = kth_brute(mat, k)
    for fn in fns:
        assert fn(mat, k) == want, (fn, mat, k)
print("ví dụ + edge + 500 ma trận ngẫu nhiên khớp brute force")

ví dụ + edge + 500 ma trận ngẫu nhiên khớp brute force


In [17]:
# kiểm tra riêng XOR tiền tố 2D với định nghĩa, và quickselect với sort trên mảng có nhiều giá trị trùng
random.seed(9)
for _ in range(200):
    m, n = random.randint(1, 8), random.randint(1, 8)
    mat = [[random.randint(0, 15) for _ in range(n)] for _ in range(m)]
    P = prefix_xor_2d(mat)
    for a in range(m):
        for b in range(n):
            x = 0
            for i in range(a + 1):
                for j in range(b + 1):
                    x ^= mat[i][j]
            assert P[a + 1][b + 1] == x
for _ in range(300):
    vals = [random.randint(0, 5) for _ in range(random.randint(1, 50))]
    k = random.randint(1, len(vals))
    assert quickselect_kth_largest(vals, k) == sorted(vals, reverse=True)[k - 1]
print("XOR tiền tố 2D khớp định nghĩa; quickselect khớp sort (kể cả giá trị trùng)")

XOR tiền tố 2D khớp định nghĩa; quickselect khớp sort (kể cả giá trị trùng)


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Copy công thức tổng tiền tố: dùng `+`/`−` thay vì `^`

In [18]:
def wrong_plus_minus(matrix, k):
    m, n = len(matrix), len(matrix[0])
    P = [[0] * (n + 1) for _ in range(m + 1)]; vals = []
    for i in range(m):
        for j in range(n):
            P[i + 1][j + 1] = P[i][j + 1] + P[i + 1][j] - P[i][j] ^ matrix[i][j]    # SAI
            vals.append(P[i + 1][j + 1])
    return sorted(vals, reverse=True)[k - 1]

for (mat, k), want in examples:
    print("sai =", wrong_plus_minus(mat, k), "| đúng =", want, "(ví dụ nhỏ có thể trùng hợp đúng)")
mat = [[1, 2], [3, 4]]
print("ma trận [[1,2],[3,4]], k=1: sai =", wrong_plus_minus(mat, 1), "| đúng =", kth_brute(mat, 1))

sai = 7 | đúng = 7 (ví dụ nhỏ có thể trùng hợp đúng)
sai = 5 | đúng = 5 (ví dụ nhỏ có thể trùng hợp đúng)
sai = 4 | đúng = 4 (ví dụ nhỏ có thể trùng hợp đúng)
ma trận [[1,2],[3,4]], k=1: sai = 3 | đúng = 4


Thêm nữa, trong Python `^` có độ ưu tiên **thấp hơn** `+`/`−`, nên `a + b - c ^ d` là `(a + b - c) ^ d`. **Sửa:** dùng `^` cho cả bốn số hạng.

### Lỗi 2 — Quên số hạng góc `P[i−1][j−1]`

In [19]:
def wrong_no_corner(matrix, k):
    m, n = len(matrix), len(matrix[0])
    P = [[0] * (n + 1) for _ in range(m + 1)]; vals = []
    for i in range(m):
        for j in range(n):
            P[i + 1][j + 1] = P[i][j + 1] ^ P[i + 1][j] ^ matrix[i][j]              # SAI: thiếu ^ P[i][j]
            vals.append(P[i + 1][j + 1])
    return sorted(vals, reverse=True)[k - 1]

for (mat, k), want in examples:
    print("sai =", wrong_no_corner(mat, k), "| đúng =", want)

sai = 7 | đúng = 7
sai = 5 | đúng = 5
sai = 5 | đúng = 4


Phần góc trên-trái bị XOR hai lần (triệt tiêu) — phải XOR lại một lần. **Sửa:** thêm `^ P[i][j]`.

### Lỗi 3 — Sắp tăng dần rồi lấy `vals[k − 1]` (ra phần tử **nhỏ** thứ `k`)

In [20]:
def wrong_ascending(matrix, k):
    P = prefix_xor_2d(matrix)
    vals = sorted(v for row in P[1:] for v in row[1:])
    return vals[k - 1]                                   # SAI: nhỏ thứ k

for (mat, k), want in examples:
    print("sai =", wrong_ascending(mat, k), "| đúng =", want)

sai = 0 | đúng = 7
sai = 4 | đúng = 5
sai = 5 | đúng = 4


**Sửa:** `sorted(..., reverse=True)[k − 1]` hoặc `vals[-k]` khi sắp tăng.

### Lỗi 4 — Heap: dùng max-heap kích thước `N` rồi pop `k` lần nhưng quên đổi dấu

In [21]:
def wrong_heap_sign(matrix, k):
    P = prefix_xor_2d(matrix)
    h = [v for row in P[1:] for v in row[1:]]
    heapq.heapify(h)                                     # SAI: heapq là MIN-heap
    for _ in range(k - 1):
        heapq.heappop(h)
    return h[0]

for (mat, k), want in examples:
    print("sai =", wrong_heap_sign(mat, k), "| đúng =", want)

sai = 0 | đúng = 7
sai = 4 | đúng = 5
sai = 5 | đúng = 4


**Sửa:** đẩy `-v` (max-heap giả lập) hoặc dùng min-heap kích thước `k` như Cách 3.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Giá trị của tọa độ = phép toán gộp trên hình chữ nhật từ góc" ⇒ **tiền tố 2D**; phép gộp là XOR thì `+` và `−` đều thành `^`.
- "Phần tử lớn thứ `k`" ⇒ sort (đơn giản), min-heap kích thước `k` (khi `k` nhỏ), quickselect (`O(N)` TB).
- Python: `sort`/`heapq.nlargest` (C) thường thắng quickselect tự viết với `N ~ 10⁶`.
- Cẩn thận độ ưu tiên toán tử `^` trong Python (thấp hơn `+`, `−`, `<<`).
- Họ bài: 304 *Range Sum Query 2D*, 1310 *XOR Queries of a Subarray*, 215 *Kth Largest Element in an Array*.

## 15. Biến thể của bài

1. **Truy vấn XOR của hình chữ nhật bất kỳ** `(r1, c1, r2, c2)`: `P[r2+1][c2+1] ^ P[r1][c2+1] ^ P[r2+1][c1] ^ P[r1][c1]`.
2. **Tổng thay vì XOR** (lớn thứ `k` của tổng tiền tố 2D): đổi `^` thành công thức cộng/trừ.
3. **Đếm số tọa độ có giá trị ≥ X**: sau khi có mảng giá trị, sort + `bisect`.
4. **Lớn thứ `k` của XOR mọi hình chữ nhật con** (không chỉ từ góc): số hình chữ nhật `O(m²n²)` — cần kỹ thuật khác (trie nhị phân cho trường hợp 1 chiều).

In [22]:
def rect_xor(P, r1, c1, r2, c2):
    """Biến thể 1."""
    return P[r2 + 1][c2 + 1] ^ P[r1][c2 + 1] ^ P[r2 + 1][c1] ^ P[r1][c1]

random.seed(3)
mat = [[random.randint(0, 100) for _ in range(6)] for _ in range(5)]
P = prefix_xor_2d(mat)
for _ in range(200):
    r1, r2 = sorted(random.sample(range(5), 2)) if random.random() < 0.8 else (2, 2)
    c1, c2 = sorted(random.sample(range(6), 2))
    x = 0
    for i in range(r1, r2 + 1):
        for j in range(c1, c2 + 1):
            x ^= mat[i][j]
    assert rect_xor(P, r1, c1, r2, c2) == x
print("biến thể 1: XOR hình chữ nhật bất kỳ bằng 4 số hạng tiền tố — OK")

biến thể 1: XOR hình chữ nhật bất kỳ bằng 4 số hạng tiền tố — OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Trực tiếp | XOR từng hình chữ nhật | `O(m²n²)` | `O(mn)` | kiểm chứng |
| 2. Tiền tố 2D + sort | bao hàm – loại trừ bằng `^` | `O(N log N)` | `O(N)` | rõ ràng |
| 3. Theo hàng + heap `k` | top-`k` | `O(N log k)` | `O(n + k)` | `k` nhỏ, tiết kiệm bộ nhớ |
| 4. Quickselect | chọn thứ `k` | `O(N)` TB | `O(N)` | lý thuyết |
| Nộp (theo hàng + sort) | `extend` + sort C | `O(N log N)` | `O(N)` | **nộp bài** |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `kthLargestValue([[1,2,3]], 1)` và `kthLargestValue([[1],[2],[3]], 3)`?

**Bài 2 (tính tay):** lập bảng giá trị mọi tọa độ của `[[3,1],[2,2]]` và tìm lớn thứ 2.

**Bài 3 (điền TODO):** hoàn thiện `all_coordinate_values(matrix)` trả về ma trận `m × n` các giá trị tọa độ, chỉ dùng `O(n)` bộ nhớ phụ ngoài kết quả.

**Bài 4 (tự viết lại):** viết `kth_nlargest(matrix, k)` dùng `heapq.nlargest`.

**Bài 5 (mini challenge):** viết `count_zero(matrix)` đếm số tọa độ `(a, b)` có giá trị **bằng 0** trong `O(mn)` thời gian và `O(n)` bộ nhớ phụ (không lưu toàn bộ giá trị), kiểm chứng bằng brute force. Mở rộng: viết `count_at_least(matrix, X)` đếm số tọa độ có giá trị `≥ X`.

In [23]:
def all_coordinate_values(matrix):
    # TODO: prev = [0]*n; với mỗi hàng: run ^= row[j]; prev[j] ^= run; lưu bản sao prev
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) Một hàng: giá trị là XOR tiền tố `1, 3, 0`. (2) Lớn nhất `3`. (3) Một cột `[1],[2],[3]`: `1, 3, 0` ⇒ lớn thứ 3 là `0`.

**Bài 2:** (1) `(0,0)=3`, `(0,1)=3^1=2`, `(1,0)=3^2=1`. (2) `(1,1)=3^1^2^2=2`. (3) Sắp `3, 2, 2, 1` ⇒ lớn thứ 2 là `2`.

**Bài 3:** (1) Dùng dạng theo hàng. (2) `out.append(prev[:])` sau mỗi hàng. (3) So với `prefix_xor_2d`.

**Bài 4:** (1) Tạo generator giá trị. (2) `heapq.nlargest(k, gen)[-1]`. (3) Kiểm tra trên các ví dụ.

**Bài 5:** (1) Dùng dạng theo hàng (`prev`, `run`) và đếm ngay khi tính `prev[j]`. (2) Không cần `vals`. (3) `count_at_least` giống hệt, đổi điều kiện thành `prev[j] >= X`; kiểm chứng: `count_at_least(M, X) ≥ k` ⇔ lớn thứ `k` `≥ X`.

## 19. Đáp án tham khảo

In [24]:
# Bài 1
assert Solution().kthLargestValue([[1, 2, 3]], 1) == 3 and Solution().kthLargestValue([[1], [2], [3]], 3) == 0

# Bài 2
assert [row[1:] for row in prefix_xor_2d([[3, 1], [2, 2]])[1:]] == [[3, 2], [1, 2]]
assert Solution().kthLargestValue([[3, 1], [2, 2]], 2) == 2

# Bài 3
def all_coordinate_values_ans(matrix):
    n = len(matrix[0]); prev = [0] * n; out = []
    for row in matrix:
        run = 0
        for j in range(n):
            run ^= row[j]; prev[j] ^= run
        out.append(prev[:])
    return out

# Bài 4
def kth_nlargest(matrix, k):
    vals = (v for row in all_coordinate_values_ans(matrix) for v in row)
    return heapq.nlargest(k, vals)[-1]

random.seed(111)
for _ in range(300):
    m, n = random.randint(1, 7), random.randint(1, 7)
    mat = [[random.randint(0, 50) for _ in range(n)] for _ in range(m)]
    assert all_coordinate_values_ans(mat) == [row[1:] for row in prefix_xor_2d(mat)[1:]]
    k = random.randint(1, m * n)
    assert kth_nlargest(mat, k) == kth_brute(mat, k)
print("Bài 1-4: OK")

Bài 1-4: OK


In [25]:
# Bài 5 — đếm tọa độ có giá trị 0 / >= X với O(n) bộ nhớ phụ
def count_at_least(matrix, X):
    n = len(matrix[0]); prev = [0] * n; c = 0
    for row in matrix:
        run = 0
        for j in range(n):
            run ^= row[j]; prev[j] ^= run
            c += prev[j] >= X
    return c

def count_zero(matrix):
    n = len(matrix[0]); prev = [0] * n; c = 0
    for row in matrix:
        run = 0
        for j in range(n):
            run ^= row[j]; prev[j] ^= run
            c += prev[j] == 0
    return c

def count_zero_brute(matrix):
    m, n = len(matrix), len(matrix[0]); c = 0
    for a in range(m):
        for b in range(n):
            x = 0
            for i in range(a + 1):
                for j in range(b + 1):
                    x ^= matrix[i][j]
            c += x == 0
    return c

random.seed(112)
for _ in range(300):
    m, n = random.randint(1, 6), random.randint(1, 6)
    mat = [[random.randint(0, 3) for _ in range(n)] for _ in range(m)]
    assert count_zero(mat) == count_zero_brute(mat)
    k = random.randint(1, m * n); kth = kth_brute(mat, k)
    assert count_at_least(mat, kth) >= k and count_at_least(mat, kth + 1) < k
print("Bài 5: OK; ví dụ [[5,2],[1,6]] có", count_zero([[5, 2], [1, 6]]), "tọa độ giá trị 0")

Bài 5: OK; ví dụ [[5,2],[1,6]] có 1 tọa độ giá trị 0


## 20. Template nhớ nhanh + Checklist

```python
# XOR tiền tố 2D theo hàng + lấy lớn thứ k
prev = [0] * n; vals = []
for row in matrix:
    run = 0
    for j in range(n):
        run ^= row[j]          # XOR hàng hiện tại từ 0..j
        prev[j] ^= run         # P[i][j] = P[i-1][j] ^ run
    vals.extend(prev)
vals.sort(reverse=True)
return vals[k - 1]
# hoặc công thức đầy đủ: P[i+1][j+1] = P[i][j+1] ^ P[i+1][j] ^ P[i][j] ^ M[i][j]
```

**Checklist trước khi submit:**
- [ ] Dùng `^` cho **cả bốn** số hạng (không `+`/`−`).
- [ ] Không quên số hạng góc `P[i][j]`.
- [ ] Lấy phần tử **lớn** thứ `k` (`reverse=True` hoặc `vals[-k]`).
- [ ] `heapq` là min-heap: dùng heap kích thước `k` hoặc đổi dấu.
- [ ] Cẩn thận độ ưu tiên `^` trong biểu thức.